# xbrl-frontier Phase 0 on a Kaggle T4 (owner-run)

Runs, in order: the base-model throughput benchmark (`serving.phase0_throughput_benchmark`), base-model quantization for the smoke test, and the T4 vLLM smoke test (`serving.smoke_test`). All numbers come from `docs/config/thresholds.yaml` inside the attached repo snapshot; nothing is set in this notebook.

**Settings:** Accelerator = GPU T4 (one GPU is used), Internet = on, attach the Kaggle dataset `xbrl-frontier-src` (the `git archive` tarball, see docs/OPEN.md).

**Download afterwards:** `/kaggle/working/phase0_kaggle_t4.tar.gz`, `/kaggle/working/smoke_kaggle_t4.tar.gz`, `/kaggle/working/smoke_checkpoints.tar.gz`.

In [ ]:
AWQ_ONLY = False   # True: only build the AWQ int4 checkpoints and smoke-test them (rerun, O-029); skips the benchmark and INT8


In [ ]:
import glob, os, shutil, subprocess, sys
dst = '/kaggle/working/xbrl-frontier'
tarballs = glob.glob('/kaggle/input/**/xbrl-frontier-src.tar.gz', recursive=True)
trees = [os.path.dirname(os.path.dirname(os.path.dirname(p)))
         for p in glob.glob('/kaggle/input/**/docs/config/thresholds.yaml', recursive=True)]
if tarballs:
    assert len(tarballs) == 1, tarballs
    os.makedirs(dst, exist_ok=True)
    subprocess.run(['tar', '-xzf', tarballs[0], '-C', dst], check=True)
else:  # Kaggle auto-extracted the upload
    assert len(trees) == 1, trees
    shutil.copytree(trees[0], dst, dirs_exist_ok=True)
os.chdir(dst)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
import yaml
pins = yaml.safe_load(open('docs/config/thresholds.yaml'))['pins']['software']
missing = [k for k in ('vllm', 'llmcompressor', 'autoawq') if not pins.get(k)]
assert not missing, f'pins.software empty for {missing}: owner approval needed first'
print(pins)

In [ ]:
!pip install -q "vllm=={pins['vllm']}" "pyarrow==25.0.1" "pyyaml==6.0.3"
!pip uninstall -y -q torchaudio  # Kaggle's torchaudio is built for another CUDA than vllm's torch; transformers imports it
!python -c "import vllm, torch; print(vllm.__version__, torch.__version__, torch.cuda.get_device_name(0))"

## 1. Base-model throughput benchmark (fp16, FinNI token shape)

In [ ]:
if not AWQ_ONLY:
    !python -m serving.phase0_benchmark --out /kaggle/working/phase0_kaggle_t4


## 2. Quantize the base students for the smoke test (never scored)
INT8 W8A8 with llm-compressor in the main environment; AWQ-INT4 with AutoAWQ in its own virtualenv because of its transformers bound.

In [ ]:
if not AWQ_ONLY:
    !pip install -q "llmcompressor=={pins['llmcompressor']}"
    !python -m serving.quantize_base --only int8 --out /kaggle/working/smoke_checkpoints


In [ ]:
!pip install -q "virtualenv==20.29.3" && python -m virtualenv -q /tmp/awqenv
!/tmp/awqenv/bin/pip install -q "autoawq=={pins['autoawq']}" "pyyaml==6.0.3" "pyarrow==25.0.1" huggingface_hub
!/tmp/awqenv/bin/python -m serving.quantize_base --only awq --out /kaggle/working/smoke_checkpoints

## 3. vLLM smoke test of every planned T4 variant

In [ ]:
_smoke = '--precision awq-int4 --out /kaggle/working/smoke_kaggle_t4_awq' if AWQ_ONLY else '--out /kaggle/working/smoke_kaggle_t4'
!python -m serving.smoke_test --card kaggle_t4 --checkpoints /kaggle/working/smoke_checkpoints {_smoke}


In [ ]:
!tar -czf /kaggle/working/smoke_checkpoints.tar.gz -C /kaggle/working smoke_checkpoints
!ls -la /kaggle/working/*.tar.gz